# Step 5 Robustness and Sensitivity Notebook

Runs predefined robustness checks and placebo test.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('../data/synthetic_session_panel.csv', parse_dates=['quote_ts_local'])

In [ ]:
def fit(formula, d):
    return smf.ols(formula, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['market_id']})

base = df[df['contaminated'] == 0].copy()
m1 = fit('conversion ~ ln_effective_price + eta_minutes + stockout_rate + rain_index + C(market_id) + C(stage) + C(hour_local)', base)
print('Baseline coef:', m1.params['ln_effective_price'])

In [ ]:
alt = base.copy()
alt['ln_total_paid'] = np.log(alt['total_paid'])
m2 = fit('conversion ~ ln_total_paid + eta_minutes + stockout_rate + rain_index + C(market_id) + C(stage) + C(hour_local)', alt)
print('Alt price coef:', m2.params['ln_total_paid'])

In [ ]:
placebo = df[df['quote_ts_local'] < '2025-02-01'].copy()
placebo['fake_post'] = (placebo['quote_ts_local'] >= '2025-01-20').astype(int)
placebo['placebo_treat'] = placebo['treated_market'] * placebo['fake_post']
m3 = fit('conversion ~ placebo_treat + eta_minutes + stockout_rate + rain_index + C(market_id) + C(stage) + C(hour_local)', placebo)
print('Placebo coef:', m3.params['placebo_treat'])